In [1]:
import os
print(os.getcwd())

C:\Users\DELL\random-forest


In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\DELL\Downloads\loan_default_data.csv")

In [3]:
print(df.shape)
print(df.columns)
df.head()

(10000, 14)
Index(['customer_id', 'age', 'annual_income', 'credit_score', 'loan_amount',
       'loan_tenure_months', 'interest_rate', 'dti', 'utilization',
       'days_past_due', 'inquiries_6m', 'existing_loans', 'employment_type',
       'default'],
      dtype='object')


,customer_id,age,annual_income,credit_score,loan_amount,loan_tenure_months,interest_rate,dti,utilization,days_past_due,inquiries_6m,existing_loans,employment_type,default
0,C00001,25,355158.0,620.0,249337,12,19.40,0.431,0.387,0,2,1,Salaried,0
1,C00002,55,482630.0,686.0,353779,36,19.36,0.440,0.054,0,2,3,Salaried,0
2,C00003,50,754891.0,662.0,541278,36,18.52,0.616,0.605,29,0,2,Salaried,0
3,C00004,40,460078.0,751.0,606237,48,14.59,0.370,0.669,92,0,2,Self-employed,0
4,C00005,40,557728.0,754.0,430453,60,14.24,0.331,0.584,21,2,1,Salaried,0


In [4]:
print("\nMissing values:")
print(df.isna().sum())


Missing values:
customer_id             0
age                     0
annual_income         200
credit_score          150
loan_amount             0
loan_tenure_months      0
interest_rate           0
dti                   100
utilization             0
days_past_due           0
inquiries_6m            0
existing_loans          0
employment_type         0
default                 0
dtype: int64


In [5]:
# Completely identical rows
print("\nDuplicate rows:", df.duplicated().sum())



Duplicate rows: 0


In [6]:
# Repeated customer IDs
print("Repeated customer IDs:", df["customer_id"].duplicated().sum())

Repeated customer IDs: 0


In [7]:
# Target counts aur percentages
print("\nTarget counts:")
print(df["default"].value_counts(dropna=False))



Target counts:
default
0    8574
1    1426
Name: count, dtype: int64


In [8]:
print("\nTarget percentage:")
print(df["default"].value_counts(normalize=True, dropna=False) * 100)


Target percentage:
default
0    85.74
1    14.26
Name: proportion, dtype: float64


In [9]:
X = df.drop(columns=["customer_id", "days_past_due", "default"])
y = df["default"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())

X shape: (10000, 11)
y shape: (10000,)
Features: ['age', 'annual_income', 'credit_score', 'loan_amount', 'loan_tenure_months', 'interest_rate', 'dti', 'utilization', 'inquiries_6m', 'existing_loans', 'employment_type']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining target percentage:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target percentage:")
print(y_test.value_counts(normalize=True) * 100)

Training shape: (8000, 11)
Test shape: (2000, 11)

Training target percentage:
default
0    85.7375
1    14.2625
Name: proportion, dtype: float64

Test target percentage:
default
0    85.75
1    14.25
Name: proportion, dtype: float64


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["employment_type"]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

print("Model training complete")

Model training complete


In [13]:
# Predicted class: 0 ya 1
y_test_pred = rf_pipeline.predict(X_test)

# Class 1 yani default ki estimated probability
y_test_prob = rf_pipeline.predict_proba(X_test)[:, 1]

prediction_preview = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": y_test_pred,
    "default_probability": y_test_prob
})

print(prediction_preview.head(10))

   actual  predicted  default_probability
0       0          0                 0.22
1       0          0                 0.14
2       0          0                 0.16
3       0          0                 0.30
4       0          0                 0.06
5       0          0                 0.04
6       0          0                 0.06
7       0          0                 0.25
8       0          0                 0.03
9       0          0                 0.43


In [14]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_train_pred = rf_pipeline.predict(X_train)
y_train_prob = rf_pipeline.predict_proba(X_train)[:, 1]

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy:", accuracy_score(y_test, y_test_pred))

print("\nTrain ROC-AUC:", roc_auc_score(y_train, y_train_prob))
print("Test ROC-AUC:", roc_auc_score(y_test, y_test_prob))

print("\nTest confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nTest classification report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

Train accuracy: 1.0
Test accuracy: 0.86

Train ROC-AUC: 1.0
Test ROC-AUC: 0.7216633420285408

Test confusion matrix:
[[1700   15]
 [ 265   20]]

Test classification report:
              precision    recall  f1-score   support

           0       0.87      0.99      0.92      1715
           1       0.57      0.07      0.12       285

    accuracy                           0.86      2000
   macro avg       0.72      0.53      0.52      2000
weighted avg       0.82      0.86      0.81      2000



In [15]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    "model__max_depth": [5, 10],
    "model__min_samples_leaf": [10, 20],
    "model__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# CV parallel chalega; har forest ek CPU job use karega
rf_pipeline.set_params(model__n_jobs=1)

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Best parameters: {'model__class_weight': 'balanced', 'model__max_depth': 10, 'model__min_samples_leaf': 20}
Best CV ROC-AUC: 0.7315304980469737


In [16]:
best_model = grid_search.best_estimator_

y_train_pred_tuned = best_model.predict(X_train)
y_train_prob_tuned = best_model.predict_proba(X_train)[:, 1]

y_test_pred_tuned = best_model.predict(X_test)
y_test_prob_tuned = best_model.predict_proba(X_test)[:, 1]

print("Train accuracy:",
      accuracy_score(y_train, y_train_pred_tuned))
print("Test accuracy:",
      accuracy_score(y_test, y_test_pred_tuned))

print("\nTrain ROC-AUC:",
      roc_auc_score(y_train, y_train_prob_tuned))
print("Test ROC-AUC:",
      roc_auc_score(y_test, y_test_prob_tuned))

print("\nTest confusion matrix:")
print(confusion_matrix(y_test, y_test_pred_tuned))

print("\nTest classification report:")
print(classification_report(
    y_test, y_test_pred_tuned, zero_division=0
))

Train accuracy: 0.80725
Test accuracy: 0.7415

Train ROC-AUC: 0.8771844639725004
Test ROC-AUC: 0.7365618126950028

Test confusion matrix:
[[1320  395]
 [ 122  163]]

Test classification report:
              precision    recall  f1-score   support

           0       0.92      0.77      0.84      1715
           1       0.29      0.57      0.39       285

    accuracy                           0.74      2000
   macro avg       0.60      0.67      0.61      2000
weighted avg       0.83      0.74      0.77      2000



In [17]:
feature_names = (
    best_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = (
    best_model.named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False)

print(importance_df)

                               feature  importance
2                    num__credit_score    0.248786
7                     num__utilization    0.185823
6                             num__dti    0.141546
5                   num__interest_rate    0.125571
3                     num__loan_amount    0.069744
1                   num__annual_income    0.063440
0                             num__age    0.054646
8                    num__inquiries_6m    0.031584
9                  num__existing_loans    0.025334
4              num__loan_tenure_months    0.019735
11       cat__employment_type_Salaried    0.016338
13     cat__employment_type_Unemployed    0.006915
10       cat__employment_type_Contract    0.005419
12  cat__employment_type_Self-employed    0.005119
